# 📐 Rotary Position Embeddings (RoPE): Complete Hands-On Lab

Welcome! In this notebook, we explore <b>RoPE (Rotary Position Embeddings)</b> through step-by-step paper-and-pencil exercises and runnable PyTorch code.

---

## 1. The Core 2D Rotation Formula

Given an initial 2D vector <b>[x₁, x₂]</b> and a rotation angle <b>θ</b>:

<pre style='font-size: 16px; background: #161b22; color: #58a6ff; padding: 12px; border-radius: 8px;'>
x'₁ = (x₁ · cos θ) - (x₂ · sin θ)
x'₂ = (x₁ · sin θ) + (x₂ · cos θ)
</pre>

On a clock face, this formula simply rotates the hand by angle <b>θ</b>.

---
## 2. Exercise 1: Single 2D Pair on Paper 📝

Let our initial vector be:
<br>
<b>x = [1.0, 0.0]</b> &nbsp; (An arrow pointing along the horizontal axis, at 3 o'clock).

<br><br>
Now let's rotate it by <b>θ = 90°</b> (a quarter turn):
- <b>cos(90°) = 0.0</b>
- <b>sin(90°) = 1.0</b>

### Step-by-Step on Paper:
1. <b>x'₁ = (1.0 · 0.0) - (0.0 · 1.0) = 0.0 - 0.0 = 0.0</b>
2. <b>x'₂ = (1.0 · 1.0) + (0.0 · 0.0) = 1.0 + 0.0 = 1.0</b>

<br>
<b>Rotated Vector [x'₁, x'₂] = [0.0, 1.0]</b><br>
*(Notice: The arrow rotated from 3 o'clock straight up to 12 o'clock!)*

In [2]:
import sys
from pathlib import Path

root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import torch
import math
from src.model.rope import apply_rotary_emb

# Input vector x = [1.0, 0.0]
x = torch.tensor([[[[1.0, 0.0]]]], dtype=torch.float32)  # [1, 1, 1, 2]

theta = math.pi / 2  # 90 degrees
cos = torch.tensor([[math.cos(theta), math.cos(theta)]])
sin = torch.tensor([[math.sin(theta), math.sin(theta)]])

x_rot, _ = apply_rotary_emb(x, x, cos, sin)
print(f"Initial Vector:  {x.squeeze().tolist()}")
print(f"Rotated Vector:  {[round(v, 4) for v in x_rot.squeeze().tolist()]}")

Initial Vector:  [1.0, 0.0]
Rotated Vector:  [0.0, 1.0]


---
## 3. Why Can't RoPE Use a Dimension of 3?

RoPE rotates in <b>2D planes</b>. A 2D plane requires <b>pairs of 2 coordinates</b> (x, y).

Because of this, in every foundation model (Llama 3, Mirendil-Base, DeepSeek):
<pre style='font-size: 15px; background: #161b22; color: #f85149; padding: 12px; border-radius: 8px;'>
The head dimension d must ALWAYS be an EVEN number (e.g. 64, 128).
An odd number like 3 would leave a number without a pair!
</pre>

A vector of dimension <b>4</b> is split into <b>two 2D pairs</b>:
<pre style='font-size: 15px; background: #161b22; color: #58a6ff; padding: 12px; border-radius: 8px;'>
x = [ x₁, x₂ ,   x₃, x₄ ]
     └ Pair 1 ┘   └ Pair 2 ┘
</pre>

---
## 4. Exercise 2: Multi-Pair Rotation (4D Vector) 📝

Let our 4D vector be:
<pre style='font-size: 16px; background: #161b22; color: #c9d1d9; padding: 12px; border-radius: 8px;'>
x = [ 1.0,  0.0,    0.0,  1.0 ]
     └─ Pair 1 ─┘  └─ Pair 2 ─┘
</pre>

Let rotation angle <b>θ = 90°</b> (cos 90° = 0.0, sin 90° = 1.0).

### Step-by-Step on Paper:

<b>Pair 1: [1.0, 0.0]</b>
- x'₁ = (1.0 · 0.0) - (0.0 · 1.0) = <b>0.0</b>
- x'₂ = (1.0 · 1.0) + (0.0 · 0.0) = <b>1.0</b>

<b>Pair 2: [0.0, 1.0]</b>
- x'₃ = (0.0 · 0.0) - (1.0 · 1.0) = 0.0 - 1.0 = <b>-1.0</b>
- x'₄ = (0.0 · 1.0) + (1.0 · 0.0) = 0.0 + 0.0 = <b>0.0</b>

<br>
<b>Final Rotated 4D Vector:</b><br>
<span style='color: #3fb950; font-size: 16px; font-weight: bold;'>
[ 0.0, 1.0,   -1.0, 0.0 ]
</span>

In [3]:
# Input 4D vector: 2 pairs
x4 = torch.tensor([[[[1.0, 0.0, 0.0, 1.0]]]], dtype=torch.float32)  # [1, 1, 1, 4]

theta = math.pi / 2
cos4 = torch.tensor(
    [[math.cos(theta), math.cos(theta), math.cos(theta), math.cos(theta)]]
)
sin4 = torch.tensor(
    [[math.sin(theta), math.sin(theta), math.sin(theta), math.sin(theta)]]
)

x4_rot, _ = apply_rotary_emb(x4, x4, cos4, sin4)
print(f"Initial 4D Vector:  {x4.squeeze().tolist()}")
print(f"Rotated 4D Vector:  {[round(v, 4) for v in x4_rot.squeeze().tolist()]}")

Initial 4D Vector:  [1.0, 0.0, 0.0, 1.0]
Rotated 4D Vector:  [0.0, 1.0, -1.0, 0.0]


---
## 5. Exercise 3: Relative Distance Invariance 🎯

Why does RoPE care only about <b>distance (m - n)</b> and not absolute positions?

Let Query token be at position <b>m</b> and Key token be at position <b>n</b>.
Their attention score is the dot product of their rotated vectors:

<pre style='font-size: 16px; background: #161b22; color: #58a6ff; padding: 12px; border-radius: 8px;'>
Score(q_m, k_n) = cos( (n - m) · θ )
</pre>

Run the cell below to test:
1. Positions (10, 12) &rarr; Distance = 2
2. Positions (40, 42) &rarr; Distance = 2

Notice that even though the absolute positions are completely different, the score is identical to 6 decimal places!

In [6]:
from src.model.rope import precompute_freqs_cis

head_dim = 64
cos_table, sin_table = precompute_freqs_cis(dim=head_dim, max_seq_len=256)
print(f"{sin_table=}, {cos_table=}")
torch.manual_seed(42)
q = torch.randn(1, 1, 1, head_dim)
k = torch.randn(1, 1, 1, head_dim)

print(f"{q=}, {k= }")
# Scenario A: Positions 10 and 12 (distance = 2)
q10, _ = apply_rotary_emb(q, q, cos_table[10:11], sin_table[10:11])
_, k12 = apply_rotary_emb(k, k, cos_table[12:13], sin_table[12:13])
score_A = (q10 * k12).sum().item()

print(f"{q10=}, {k12= }")

# Scenario B: Positions 40 and 42 (distance = 2)
q40, _ = apply_rotary_emb(q, q, cos_table[40:41], sin_table[40:41])
_, k42 = apply_rotary_emb(k, k, cos_table[42:43], sin_table[42:43])
score_B = (q40 * k42).sum().item()
print(f"{q40=}, {k42= }")

print(f"Attention Score at positions (10, 12): {score_A:.6f}")
print(f"Attention Score at positions (40, 42): {score_B:.6f}")
print(f"Absolute Difference:                   {abs(score_A - score_B):.10f}")

assert abs(score_A - score_B) < 1e-5
print("\nSUCCESS: Relative distance invariance holds perfectly!")

sin_table=tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 8.4147e-01,  8.4147e-01,  6.8156e-01,  ...,  1.7783e-04,
          1.3335e-04,  1.3335e-04],
        [ 9.0930e-01,  9.0930e-01,  9.9748e-01,  ...,  3.5566e-04,
          2.6670e-04,  2.6670e-04],
        ...,
        [ 9.9482e-01,  9.9482e-01,  9.4171e-01,  ...,  4.4975e-02,
          3.3732e-02,  3.3732e-02],
        [ 4.5200e-01,  4.5200e-01,  9.1840e-01,  ...,  4.5153e-02,
          3.3865e-02,  3.3865e-02],
        [-5.0639e-01, -5.0639e-01,  4.0239e-01,  ...,  4.5331e-02,
          3.3998e-02,  3.3998e-02]]), cos_table=tensor([[ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000],
        [ 0.5403,  0.5403,  0.7318,  ...,  1.0000,  1.0000,  1.0000],
        [-0.4161, -0.4161,  0.0709,  ...,  1.0000,  1.0000,  1.0000],
        ...,
        [-0.1016, -0.1016,  0.3364,  ...,  0.9990,  0.9994,  0.9994],
        [-0.8920, -0.8920, -0.3956,  ...,  0.9990,  0.